# Canonical Model 06: Ensemble Uncertainty & IES Diagnostics

This notebook takes the same calibration setup as Notebook 04 and works through the **per-cycle ensemble diagnostics** a modeller should look at every time: prior Monte Carlo and prior-data conflict, phi distribution, phi contribution by observation group, parameters at bounds, and posterior forecast uncertainty.

Every plot offers both a `plotly` and a `matplotlib` backend and a short "what to look for".

In [ ]:
import sys
from pathlib import Path

# Make the in-repo `src/` importable when myflopy is not pip-installed.
src = Path.cwd().parents[2] / "src"
if src.exists() and str(src) not in sys.path:
    sys.path.insert(0, str(src))

import pandas as pd
import myflopy as mf
from canonical_notebook_style import notebook_header
from myflopy.modflow.mf6.canonical_calibration import build_canonical_calibration_demo
from myflopy.modflow.mf6.pest import PestProject

notebook_header('06', 'Ensemble Uncertainty & IES Diagnostics',
                'Prior Monte Carlo, prior-data conflict, phi & weight diagnostics, forecast uncertainty.')

## 1. Build and declare the PEST problem (same as Notebook 04)

In [ ]:
artifact_root = Path('../artifacts/canonical_pest')
artifact_root.mkdir(parents=True, exist_ok=True)

demo = build_canonical_calibration_demo(artifact_root / 'model')

cal = PestProject(model=demo.model, name='canonical_pest',
                  workspace=artifact_root / 'template', start_datetime='2024-01-01')
# One constant K multiplier (scales every layer's K file together) and one
# constant recharge multiplier. Bounds are multiplier factors; `physical`
# clamps the final model value so calibration cannot reach nonphysical K.
cal.parameterize('k',        style='constant', bounds=(0.05, 2.0), physical=(0.01, 300.0))
cal.parameterize('recharge', style='constant', bounds=(0.3, 3.0),  physical=(0.0, 1e-2))
cal.observe(demo.head_targets)
cal.forecast(demo.forecast_targets)
pst = cal.build('canonical_pest.pst', noptmax=0)
print(cal.settings())

## 2. Prior Monte Carlo -- "early and often"

Before history matching, evaluate the **prior** parameter ensemble once (`cal.prior(...)`, NOPTMAX = -1). This is cheap (a single ensemble, no iterations) and answers two questions: is the prior wide enough to explain the data, and is any observation *outside* what the prior can produce? The grey "spaghetti" is the prior simulated ensemble; red triangles are the measured values. **What to look for:** measured points that fall inside the grey band (the prior can reach them).

In [ ]:
pm = cal.prior(reals=50, workers=12)
display(pm.plot_prior_vs_obs())                      # grey prior spaghetti vs measured
display(pm.plot_prior_vs_obs(backend='matplotlib'))

## 3. Prior-data conflict

Where a measured value lies *outside* the prior predictive band, the prior and the data disagree before calibration even starts -- a **prior-data conflict**. That usually means the prior is too narrow, a weight/observation is wrong, or the model is structurally missing something. **What to look for:** groups with a high % in conflict; investigate them before trusting history matching.

In [ ]:
conflict = pm.conflict()
print('observations in conflict:', int(conflict['in_conflict'].sum()), 'of', len(conflict))
display(pm.plot_conflict())
display(conflict[conflict['in_conflict']].head(10))

## 4. Run PESTPP-IES

A modest ensemble with parallel agents (slow: ~minutes). Set `RUN_IES = False` to skip.

In [ ]:
RUN_IES = True
if RUN_IES:
    ies = cal.run_ies(reals=30, iterations=3, workers=12)
    print(ies.settings)
else:
    ies = None
    print('Skipped: set RUN_IES = True to run PESTPP-IES.')

## 5. Phi convergence and distribution (the "pepsi challenge")

`plot_phi` shows the objective function dropping across iterations. `plot_phi_distribution` overlays the prior and posterior phi *histograms* (log scale) -- the workshop's "pepsi challenge". **What to look for:** the posterior histogram shifted left (lower phi) of the prior, and a posterior phi near the number of nonzero observations (a phi far below that is over-fitting).

In [ ]:
if ies is not None:
    display(ies.plot_phi())
    display(ies.plot_phi_distribution())                     # plotly
    display(ies.plot_phi_distribution(backend='matplotlib'))

## 6. Phi contribution by observation group

Which observation groups dominate the misfit? `plot_phi_contributions` breaks the total phi down by group. **What to look for:** one group swamping the others usually means its weights are too high relative to the rest -- rebalance and rerun rather than letting it drive the solution.

In [ ]:
if ies is not None:
    display(ies.plot_phi_contributions())
    display(ies.plot_phi_contributions(kind='pie', backend='matplotlib'))

## 7. Parameters at their bounds

A parameter pinned at a bound is calibration "asking for more" than the prior allows -- often a sign the bound is too tight or the parameterization is too stiff. `parameters_at_bounds` reports the fraction of each parameter group sitting at a bound. **What to look for:** groups with a high % at bound; consider widening bounds or adding flexibility.

In [ ]:
if ies is not None:
    display(ies.parameters_at_bounds())
    display(ies.plot_parameters_at_bounds())

## 8. Forecast uncertainty -- "uncertainty analysis for free"

The payoff of IES: the posterior distribution of the prediction of interest (a down-valley head near the lake). `forecast(name).plot()` shows prior (grey) and posterior (blue) histograms. **What to look for:** history matching should *narrow* the forecast relative to the prior -- but a good fit is not the same as a good prediction, so value the posterior spread, not just a reduced phi.

In [ ]:
if ies is not None:
    display(ies.forecasts())
    name = ies.forecast_names[0]
    display(ies.forecast(name).plot())
    display(ies.forecast(name).plot(backend='matplotlib'))

## 9. The single best realization, and the report bundle

Carry the **base** (minimum-error-variance) realization forward, never the lowest-phi one (it tends to be over-fit). `report(...)` bundles the diagnostics into one HTML file.

In [ ]:
if ies is not None:
    print('recommended realization:', ies.best())
    report_path = ies.report(artifact_root / 'uncertainty_review.html')
    print('wrote', report_path)

## Interpretation checklist (run this every cycle)

- **Prior Monte Carlo first:** is the prior wide enough, and is any observation in conflict with it?
- **Phi:** did it drop, and is the posterior phi near (not far below) the observation count?
- **Phi by group:** is one group dominating? Rebalance weights if so.
- **Parameters at bounds:** are bounds too tight?
- **Forecast:** value the posterior *spread* -- a good fit is not a good prediction.
- Carry the **base** realization forward.